# EDA and Text Preprocessing — Urgency Classification

Pipeline persiapan data untuk klasifikasi tingkat urgensi komplain konsumen.

## Alur Pipeline
1. Load dataset mentah
2. Pembersihan data awal (filter invalid)
3. Auto-labeling kolom `Urgency`
4. Text preprocessing & normalisasi
5. EDA — distribusi kelas & fitur struktural
6. EDA NLP — N-Gram & Word Cloud
7. Simpan dataset bersih

## Tahap 1 — Setup

- Deteksi lingkungan (Colab / lokal) dan atur `PROJECT_ROOT`
- Tambahkan root ke `sys.path` agar modul `src` dapat diimport

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/lentera-analytics-hub/lentera-ml-research'
    if not os.path.exists(PROJECT_ROOT):
        PROJECT_ROOT = '/content/drive/MyDrive/lentera-ml-research'
    os.chdir(PROJECT_ROOT)
else:
    notebook_dir = os.getcwd()
    if os.path.basename(notebook_dir) == 'notebooks':
        PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, '..'))
    elif os.path.exists(os.path.join(notebook_dir, 'notebooks')):
        PROJECT_ROOT = notebook_dir
    else:
        PROJECT_ROOT = os.path.abspath(os.path.join(notebook_dir, '..'))
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

sns.set_theme(style="whitegrid")
print(f"Project root: {PROJECT_ROOT}")

## Tahap 2 — Data: Load, Bersihkan & Labeling

### 1. Load Data

- Baca dataset mentah dari `data/raw/`
- Tampilkan shape dan sampel awal

In [ ]:
df = pd.read_csv('data/raw/complaints-2026-06-19_03_14.csv')
print(f"Shape dataset: {df.shape}")
df.head(2)

### 2. Pembersihan Data Awal

- Filter baris tidak valid menggunakan `filter_invalid_complaints` dari `src.preprocessing`
- Menghapus entri duplikat, kosong, dan tidak relevan

In [ ]:
from src.preprocessing import filter_invalid_complaints

df = filter_invalid_complaints(df)
print(f"Shape setelah pembersihan awal: {df.shape}")

### 3. Auto-Labeling Urgency

- Buat kolom `Urgency` secara otomatis berdasarkan aturan berbasis fitur
- Tampilkan distribusi kelas hasil labeling

In [ ]:
from src.preprocessing import apply_urgency_labels

df = apply_urgency_labels(df)

print("Distribusi Urgency:")
print(df['Urgency'].value_counts())

### 4. Text Preprocessing

- Normalisasi teks: lowercase, hapus karakter khusus, stopword removal, lemmatisasi
- Hasil disimpan ke kolom `Cleaned_Narrative`

In [ ]:
from src.preprocessing import normalize_text_pipeline

df = normalize_text_pipeline(df)

## Tahap 3 — EDA: Distribusi & Fitur Struktural

### 5. Exploratory Data Analysis

- Visualisasi distribusi kelas `Urgency` (bar chart & pie chart)
- Analisis hubungan `Urgency` dengan `Company public response` dan `Timely response?`
- Distribusi panjang kata per kelas

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='Urgency', data=df, order=['High', 'Medium', 'Low'], palette='viridis')
plt.title('Distribusi Kelas Urgensi')
plt.ylabel('Jumlah Komplain')
plt.xlabel('Tingkat Urgensi')
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
class_counts = df['Urgency'].value_counts()
colors = ['#ff4d4d', '#ffa64d', '#66b3ff']
plt.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', startangle=90,
        colors=colors, wedgeprops={'edgecolor': 'black'})
plt.title('Distribusi Kelas Urgensi (Pie Chart)', fontsize=14, fontweight='bold')
plt.axis('equal')
plt.show()

#### EDA: Urgency vs. `Company public response`

In [ ]:
df['Response_Missing'] = df['Company public response'].isna() | (df['Company public response'] == 'None')
missing_stats = df.groupby('Urgency')['Response_Missing'].value_counts(normalize=True).unstack() * 100

plt.figure(figsize=(8, 5))
missing_stats.plot(kind='bar', stacked=True, colormap='Set2', ax=plt.gca())
plt.title('Proporsi Data Kosong pada Company Public Response per Urgency')
plt.ylabel('Persentase (%)')
plt.xlabel('Tingkat Urgensi')
plt.legend(title='Response Missing?', labels=['Ada Response', 'Kosong ("None")'],
           bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import textwrap

df_with_response = df[~df['Response_Missing']]
crosstab_response = pd.crosstab(
    df_with_response['Company public response'],
    df_with_response['Urgency'],
    normalize='index'
) * 100

if set(['High', 'Medium', 'Low']).issubset(crosstab_response.columns):
    crosstab_response = crosstab_response[['High', 'Medium', 'Low']]

plt.figure(figsize=(10, 8))
crosstab_response.plot(kind='barh', stacked=True, colormap='coolwarm_r', ax=plt.gca())
plt.title('Proporsi Tingkat Urgensi Berdasarkan Jenis Respons Perusahaan')
plt.xlabel('Persentase (%)')
plt.ylabel('Jenis Respons')
plt.legend(title='Urgency', bbox_to_anchor=(1.05, 1), loc='upper left')
ylabels = [textwrap.fill(label.get_text(), 40) for label in plt.gca().get_yticklabels()]
plt.gca().set_yticklabels(ylabels)
plt.tight_layout()
plt.show()

#### EDA: Urgency vs. `Timely response?`

In [ ]:
if 'Timely response?' in df.columns:
    timely_urgency_stats = pd.crosstab(df['Urgency'], df['Timely response?'], normalize='index') * 100

    if set(['High', 'Medium', 'Low']).issubset(timely_urgency_stats.index):
        timely_urgency_stats = timely_urgency_stats.loc[['High', 'Medium', 'Low']]

    if set(['No', 'Yes']).issubset(timely_urgency_stats.columns):
        timely_urgency_stats = timely_urgency_stats[['No', 'Yes']]
        warna_custom = ['#ff9999', '#66b3ff']
    else:
        warna_custom = None

    plt.figure(figsize=(8, 6))
    timely_urgency_stats.plot(kind='bar', stacked=True, color=warna_custom, ax=plt.gca())
    plt.title('Proporsi Ketepatan Waktu Respons Berdasarkan Tingkat Urgensi')
    plt.ylabel('Persentase (%)')
    plt.xlabel('Tingkat Urgensi')
    plt.xticks(rotation=0)
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(list(reversed(handles)), list(reversed(labels)),
               title='Timely Response?', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("Kolom 'Timely response?' tidak ditemukan di dataframe.")

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='Urgency', y='Word Count', data=df, order=['High', 'Medium', 'Low'], palette='Set2')
plt.title('Distribusi Panjang Kata (Word Count) Berdasarkan Tingkat Urgensi')
plt.ylim(0, 500)
plt.ylabel('Jumlah Kata')
plt.xlabel('Tingkat Urgensi')
plt.show()

## Tahap 4 — EDA NLP: N-Gram & Word Cloud

### 6. Analisis N-Gram (Bigram)

- Ekstrak 15 bigram paling sering per kelas urgensi
- Menggunakan `CountVectorizer` dengan `stop_words='english'`

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

def plot_top_ngrams(text_series, n, title, color):
    vec = CountVectorizer(ngram_range=(n, n), stop_words='english').fit(text_series)
    bag_of_words = vec.transform(text_series)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = sorted(
        [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()],
        key=lambda x: x[1], reverse=True
    )[:15]
    df_ngram = pd.DataFrame(words_freq, columns=['N-Gram', 'Frekuensi'])
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Frekuensi', y='N-Gram', data=df_ngram, color=color)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel('Frekuensi', fontsize=12)
    plt.ylabel(f'{n}-Gram', fontsize=12)
    plt.tight_layout()
    plt.show()

high_texts   = df[df['Urgency'] == 'High']['Cleaned_Narrative']
medium_texts = df[df['Urgency'] == 'Medium']['Cleaned_Narrative']
low_texts    = df[df['Urgency'] == 'Low']['Cleaned_Narrative']

plot_top_ngrams(high_texts,   2, 'Top 15 Bigrams - High Urgency',   'crimson')
plot_top_ngrams(medium_texts, 2, 'Top 15 Bigrams - Medium Urgency', 'orange')
plot_top_ngrams(low_texts,    2, 'Top 15 Bigrams - Low Urgency',    'mediumseagreen')

### Word Cloud per Kelas Urgensi

- Visualisasi kata-kata dominan per kelas (High / Medium / Low)
- Warna disesuaikan untuk mencerminkan tingkat urgensi

In [ ]:
from wordcloud import WordCloud

def plot_wordcloud(text_series, title, colormap):
    text = " ".join(text_series.astype(str))
    wordcloud = WordCloud(
        width=800, height=400,
        background_color='white',
        colormap=colormap,
        max_words=100,
        contour_width=3,
        contour_color='steelblue'
    ).generate(text)
    plt.figure(figsize=(12, 6))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=18, fontweight='bold', pad=20)
    plt.show()

plot_wordcloud(high_texts,   'Word Cloud - HIGH Urgency',   'Reds')
plot_wordcloud(medium_texts, 'Word Cloud - MEDIUM Urgency', 'Oranges')
plot_wordcloud(low_texts,    'Word Cloud - LOW Urgency',    'Greens')

## Tahap 5 — Simpan Dataset Bersih

### 7. Ekspor Dataset Final

- Hapus baris kosong yang tersisa pasca-preprocessing
- Simpan kolom `Raw_Filtered_Narrative`, `Cleaned_Narrative`, dan `Urgency` ke `data/processed/cleaned_complaints.csv`

In [ ]:
df = df.dropna(subset=['Cleaned_Narrative', 'Raw_Filtered_Narrative'])
df = df[df['Cleaned_Narrative'].str.strip() != '']

df_final = df[['Raw_Filtered_Narrative', 'Cleaned_Narrative', 'Urgency']].copy()
df_final.to_csv('data/processed/cleaned_complaints.csv', index=False)

print(f"Shape dataset final: {df_final.shape}")
df_final.head(3)